# 00 · Join UBIGEO y carga de fuentes de datos

Notebook de la Etapa 0 del pipeline: antes de tocar una sola imagen, se cargan y verifican todas las fuentes de datos y se construye la tabla maestra unida por UBIGEO.

## Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import requests

pd.set_option("display.max_columns", 50)


# Polígonos y ubicación geográfica

## Límite distrital INEI 2025 (shapefile)

Fuente: https://www.geogpsperu.com/2019/05/limite-distrital-actualizado-inei.html

Descarga manual (no tiene link directo de descarga) — bajar el shapefile y colocar la ruta local abajo.

In [ ]:
# Ruta local al shapefile ya descargado
ruta_limite_distrital = ""  # ej: "data/raw/Limite_Distrital_INEI_2025_CPV.shp"

limite_distrital = gpd.read_file(ruta_limite_distrital)
print(limite_distrital.shape)
limite_distrital.head()


## Tabla de UBIGEO (crosswalk departamento-provincia-distrito)

Fuente (CSV público, se puede leer directo desde la URL):
https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv

In [ ]:
url_ubigeo = "https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv"

ubigeo = pd.read_csv(url_ubigeo)
print(ubigeo.shape)
ubigeo.head()


# Imágenes y variables satelitales (Google Earth Engine)

## Inicializar Earth Engine

Requiere cuenta de Google Earth Engine aprobada: https://earthengine.google.com

In [ ]:
import ee

# ee.Authenticate()  # solo la primera vez
ee.Initialize()


## VIIRS Nighttime Lights

Dataset GEE: `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG`

In [ ]:
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG").select("avg_rad")
viirs_mean = viirs.mean()
print(viirs.first().getInfo())


## ESA WorldCover

Dataset GEE: `ESA/WorldCover/v200` — cobertura de suelo (% construido, % cultivo, % árboles).

In [ ]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
print(worldcover.bandNames().getInfo())


## SRTM — elevación y pendiente del terreno

Dataset GEE: `USGS/SRTMGL1_003`

In [ ]:
srtm = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(srtm)
print(srtm.bandNames().getInfo())


## Sentinel-2 L2A

Dataset GEE: `COPERNICUS/S2_SR_HARMONIZED` — serie temporal, 12 bandas, 10m.

In [ ]:
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2024-01-01", "2024-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
)
print("Número de imágenes:", sentinel2.size().getInfo())


## Planet NICFI

Registro gratuito para el trópico: https://www.planet.com/nicfi/

Requiere API key propia de Planet, no se accede vía Earth Engine público.

In [ ]:
planet_api_key = ""  # pegar aquí la API key de Planet NICFI

headers = {"Authorization": f"api-key {planet_api_key}"}
url_nicfi = "https://api.planet.com/basemaps/v1/mosaics"

# response = requests.get(url_nicfi, headers=headers)
# response.json()


## Google Open Buildings V3

Fuente: https://sites.research.google/gr/open-buildings/

Se puede acceder también como tabla en Earth Engine: `GOOGLE/Research/open-buildings/v3/polygons`

In [ ]:
open_buildings = ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")
print(open_buildings.limit(5).getInfo())


# Datos administrativos y municipales

## RENAMU (Registro Nacional de Municipalidades)

Fuente: https://proyectos.inei.gob.pe/microdatos/ (buscar "RENAMU", módulo usado antes: 1963)

In [ ]:
ruta_renamu = ""  # ej: "data/raw/RENAMU_1963.csv"

renamu = pd.read_csv(ruta_renamu)
print(renamu.shape)
renamu.head()


## ENAHO (Encuesta Nacional de Hogares)

Fuente: https://proyectos.inei.gob.pe/microdatos/ (buscar "ENAHO", módulos 1, 34, 200, 300, 400, 500)

In [ ]:
ruta_enaho_hogar = ""    # ej: "data/raw/ENAHO_1_hogar.csv"
ruta_enaho_sumaria = ""  # ej: "data/raw/ENAHO_34_sumaria.csv"

enaho_hogar = pd.read_csv(ruta_enaho_hogar)
enaho_sumaria = pd.read_csv(ruta_enaho_sumaria)

print(enaho_hogar.shape, enaho_sumaria.shape)
enaho_hogar.head()


## ENDES (Encuesta Demográfica y de Salud Familiar)

Fuente: https://proyectos.inei.gob.pe/microdatos/ (buscar "ENDES", módulos RECH0 y RECH23)

In [ ]:
ruta_endes_rech0 = ""   # ej: "data/raw/ENDES_1629_RECH0.csv"
ruta_endes_rech23 = ""  # ej: "data/raw/ENDES_1630_RECH23.csv"

endes_rech0 = pd.read_csv(ruta_endes_rech0)
endes_rech23 = pd.read_csv(ruta_endes_rech23)

print(endes_rech0.shape, endes_rech23.shape)
endes_rech0.head()


# Anemia (registro administrativo)

## SIEN / REUNIS (MINSA)

Fuente: https://www.minsa.gob.pe/reunis/

No tiene descarga directa en CSV — requiere revisar el tablero o solicitar la data en otro formato.

In [ ]:
ruta_sien = ""  # ej: "data/raw/sien_hisminsa_anemia.csv"

sien = pd.read_csv(ruta_sien)
print(sien.shape)
sien.head()


## Datos Abiertos — Anemia (MINSA / INS)

Fuente: https://datosabiertos.gob.pe/dataset/anemia

In [ ]:
ruta_anemia_datosabiertos = ""  # ej: "data/raw/anemia_datosabiertos.csv"

anemia_da = pd.read_csv(ruta_anemia_datosabiertos)
print(anemia_da.shape)
anemia_da.head()


# Presupuesto público

## SIAF Consulta Amigable (MEF)

Fuente: https://apps5.mineco.gob.pe/transparencia/Navegador/default.aspx

No tiene descarga directa — requiere scraping (ASP.NET, viewstate). Colocar aquí el CSV ya scrapeado.

In [ ]:
ruta_siaf = ""  # ej: "data/raw/siaf_gasto_distrital.csv"

siaf = pd.read_csv(ruta_siaf)
print(siaf.shape)
siaf.head()


# Vías

## OpenStreetMap / Overpass API

Fuente: https://overpass-api.de

Ejemplo de consulta para red vial de un departamento (ajustar el área de interés).

In [ ]:
overpass_url = "https://overpass-api.de/api/interpreter"

overpass_query = """
[out:json][timeout:60];
area["name"="Huancavelica"]->.searchArea;
(
  way["highway"](area.searchArea);
);
out geom;
"""

# response = requests.get(overpass_url, params={"data": overpass_query})
# vias = response.json()


# Join final por UBIGEO

Una vez cargadas todas las fuentes, se unen por la llave `ubigeo` para construir la tabla maestra distrital.

In [ ]:
# maestro_distritos = (
#     ubigeo
#     .merge(renamu, on="ubigeo", how="left")
#     .merge(enaho_sumaria, on="ubigeo", how="left")
#     .merge(endes_rech0, on="ubigeo", how="left")
#     .merge(sien, on="ubigeo", how="left")
#     .merge(anemia_da, on="ubigeo", how="left")
#     .merge(siaf, on="ubigeo", how="left")
# )
#
# print("Distritos en maestro:", maestro_distritos.shape[0])
# print("Duplicados UBIGEO:", maestro_distritos["ubigeo"].duplicated().sum())
# maestro_distritos.head()


## Guardar tabla maestra

In [ ]:
ruta_salida = ""  # ej: "data/clean/maestro_distritos_2025.csv"

# maestro_distritos.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
# print("Archivo guardado en:", ruta_salida)
